# Anatomy of a Redemption
### Spark-basierte Multi-Source Sentiment-Analyse von Cyberpunk 2077

Lab Group 1 — Data Processing 2, SS 2026

Batch-Pipeline: Reddit-Kommentare + Steam-Reviews ueber das 34-Monats-Fenster
(Dez 2020 – Dez 2023), Sentiment via VADER, Ausrichtung gegen eine Patch-Timeline.
**Kernthese:** Reddit erholt sich schneller als Steam.

**Stages:** 0 Konfiguration · 1 Ingest+Filter · 2 Cleaning · 3 VADER · 4 Aggregation · 5 Analyse/Plots

> Finale, aggregierte Ergebnisse landen in `results/` (klein, im Repo). Der Report wird aus
> diesen Dateien geschrieben, nicht durch erneuten Pipeline-Lauf.

## 0 · Konfiguration & Parameter (zentral)

Alle Stellschrauben in **einem Block** — Voraussetzung fuer Reproduzierbarkeit. `DATA_MODE`
ist der einzige Hebel zwischen Sample (Entwicklung) und vollem Lauf (`results/`-Stand). `SEED`
deckt alles Stochastische ab (v.a. LDA).

In [ ]:
import findspark
findspark.init()

from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ===== Zentrale Parameter (ein Block — Reproduzierbarkeit) =====
DATA_MODE = "sample"            # "sample" | "full"
SEED = 42                       # fuer alles Stochastische (v.a. LDA setSeed)
assert DATA_MODE in ("sample", "full")

WINDOW_START = "2020-12-01"     # 34-Monats-Fenster, inklusive Grenzen
WINDOW_END   = "2023-12-31"
SUBREDDITS   = ["cyberpunkgame", "LowSodiumCyberpunk", "Games", "pcgaming"]
STEAM_APP_ID = 1091500          # Cyberpunk 2077

ANOMALY_Z_THRESHOLD = 3.0       # Volume-Anomaly / Review-Bombing-Flag
LDA_NUM_TOPICS = 8              # Topic-Analyse

# ===== Pfade =====
_SUB       = "sample" if DATA_MODE == "sample" else "raw"
RAW_REDDIT = f"data/{_SUB}/reddit/"
RAW_STEAM  = f"data/{_SUB}/steam/"
PROCESSED  = "data/processed/"
RESULTS    = "results/"
FIGURES    = "results/figures/"
Path(RESULTS).mkdir(exist_ok=True)
Path(FIGURES).mkdir(parents=True, exist_ok=True)

print(f"DATA_MODE={DATA_MODE}  Fenster={WINDOW_START}..{WINDOW_END}  Seed={SEED}")

### SparkSession
Lokal, `local[*]`. Treiber-RAM fuer den `full`-Lauf erhoeht.

In [ ]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("CyberpunkSentiment")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "200")          # bei vollem Lauf ggf. erhoehen
    .config("spark.driver.memory", "8g" if DATA_MODE == "sample" else "12g")
    .config("spark.sql.files.maxPartitionBytes", "128m")    # Partitionsgroesse steuern
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

## 1 · Ingest + Sofort-Filter

**Harte Regel:** sofort beim Einlesen filtern (Subreddits + Zeitfenster), bevor irgendetwas
anderes passiert. Aus GB/Monat soll frueh ein Bruchteil werden. `.zst`-Dumps liest Spark direkt.
Spaltennamen (v.a. Steam) vor dem Hardcoden gegen das konkrete Dataset pruefen.

Idee: Spark haelt NIE den ganzen Datensatz im RAM — liest partitioniert von der Platte, filtert
sofort, schreibt ein viel kleineres Parquet zurueck. Dieselbe Pipeline laeuft auf Sample wie auf
vollem Fenster.

In [ ]:
# -------------------------------------------------------------
# 1) Reddit einlesen  (.zst, zeilenweise JSON — Spark liest direkt)
# -------------------------------------------------------------
reddit = (
    spark.read.json(RAW_REDDIT)                    # liest alle Dateien partitioniert
    .select("subreddit", "body", "created_utc")    # nur benoetigte Spalten
    .filter(F.col("subreddit").isin(SUBREDDITS))   # auf unsere Subreddits
    .filter(F.col("body").isNotNull())
    .filter(F.col("body") != "[deleted]")
    .filter(F.col("body") != "[removed]")
)

# Zeitfenster (created_utc ist Unix-Sekunden)
START = F.unix_timestamp(F.lit(WINDOW_START), "yyyy-MM-dd")
END   = F.unix_timestamp(F.lit(WINDOW_END), "yyyy-MM-dd")
reddit = reddit.filter((F.col("created_utc") >= START) & (F.col("created_utc") <= END))

# nutzbares Datum + Monatsbucket fuer spaetere Zeitreihen
reddit = (
    reddit
    .withColumn("date", F.to_date(F.from_unixtime("created_utc")))
    .withColumn("month", F.date_format("date", "yyyy-MM"))
    .withColumn("source", F.lit("reddit"))
    .withColumnRenamed("body", "text")
    .select("source", "month", "date", "text")
)

# -------------------------------------------------------------
# 2) Steam-Reviews einlesen (CSV)
# -------------------------------------------------------------
steam = (
    spark.read.option("header", True).option("multiLine", True).csv(RAW_STEAM)
    .select(
        F.col("review").alias("text"),
        F.col("timestamp_created").cast("long").alias("created_utc"),
    )
    .filter(F.col("text").isNotNull())
)
steam = (
    steam
    .filter((F.col("created_utc") >= START) & (F.col("created_utc") <= END))
    .withColumn("date", F.to_date(F.from_unixtime("created_utc")))
    .withColumn("month", F.date_format("date", "yyyy-MM"))
    .withColumn("source", F.lit("steam"))
    .select("source", "month", "date", "text")
)

# -------------------------------------------------------------
# 3) Vereinen + als Parquet speichern (partitioniert nach source + month)
# -------------------------------------------------------------
combined = reddit.unionByName(steam)

(
    combined.write
    .mode("overwrite")
    .partitionBy("source", "month")
    .parquet(PROCESSED + "cleaned/")
)
print("Cleaning fertig. Geschrieben nach:", PROCESSED + "cleaned/")

# Verifikation auf aggregierter Ebene (kein collect/toPandas auf dem vollen df!)
combined.groupBy("source").count().show()

# -------------------------------------------------------------
# Ab hier weiterarbeiten mit:  df = spark.read.parquet(PROCESSED + "cleaned/")
# Das laedt nur die kleinen, gefilterten Daten — schnell.
# NIEMALS .collect() / .toPandas() auf dem vollen df! Erst aggregieren,
# DANN toPandas() auf dem winzigen Ergebnis fuer die Plots.
# -------------------------------------------------------------

### Run-Info festhalten
Welcher Modus / welches Fenster liegt dem `results/`-Stand zugrunde (Reproduzierbarkeit).

In [ ]:
from datetime import datetime, timezone

Path(RESULTS, "RUN_INFO.txt").write_text(
    f"DATA_MODE = {DATA_MODE}\n"
    f"window    = {WINDOW_START} .. {WINDOW_END}\n"
    f"subreddits= {SUBREDDITS}\n"
    f"seed      = {SEED}\n"
    f"generated = {datetime.now(timezone.utc).isoformat()}\n"
)
print((Path(RESULTS) / "RUN_INFO.txt").read_text())

## 2 · Cleaning & Preprocessing  *(Phase 2 — TODO)*

- Text normalisieren (Lowercasing, URLs/Markdown raus, Whitespace)
- Sprachfilter (nur EN — VADER ist auf Englisch trainiert)
- Dedupe

## 3 · VADER-Sentiment-UDF  *(Phase 3 — TODO)*

VADER als Spark-UDF auf alle Comments/Reviews; `compound`-Score behalten.

## 4 · Aggregation je Monat/Source  *(Phase 3 — TODO)*

Sentiment je `month` x `source` aggregieren — **erst hier** wird es klein genug fuer `.toPandas()`.

Schreibt: `results/sentiment_timeseries.csv` (`source, month, mean_sentiment, n`).

## 5 · Analyse & Visualisierung  *(Phase 3 — TODO)*

Jede Ausgabe -> genau eine Datei unter `results/` (siehe Report-Mapping in der README):

- Zwei Sentiment-Zeitreihen + Patch-Marker  -> `results/figures/`, `results/patch_overlay.csv`
- Review-Bombing-Anomalie-Flag (Volume-Anomaly, `ANOMALY_Z_THRESHOLD`)  -> `results/anomaly_flags.csv`
- Sentiment ohne geflaggte Fenster  -> `results/sentiment_excl_bombing.csv`
- Recovery-Metrik (Kernthese)  -> `results/recovery_metric.csv`
- Topic-Analyse (TF-IDF / LDA mit `setSeed(SEED)`, `LDA_NUM_TOPICS`)  -> `results/topics.csv`